# ECN 4199 Week 7 Live Notebook

## Regression Analysis and Predictive Modeling for Economics

This notebook follows the Week 7 lecture step by step. It imports the course dataset directly from GitHub, so students can open the notebook in Google Colab and run the examples without manually uploading the CSV file.

**Main learning idea:** regression is not just code. It is a way of thinking carefully about economic relationships, predictions, errors, and limitations.

## 1. Setup

Run this cell first. We use `pandas` for data handling, `matplotlib` for visualization, `statsmodels` for econometric regression output, and `scikit-learn` for a prediction-style comparison.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

plt.style.use("seaborn-v0_8-whitegrid")

## 2. Import the Week 7 Dataset

The data are imported directly from the course GitHub repository. This is the easiest method in Google Colab because students do not need to upload files manually.

Each row represents one hypothetical household.

In [ ]:
url = "https://raw.githubusercontent.com/kanishkawerawella/ECN4199_Website/main/data/consumption_data.csv"
data = pd.read_csv(url)

data.head()

## 3. Understand the Variables

| Variable | Meaning |
|---|---|
| `Household` | Household identifier |
| `Income` | Monthly household income |
| `Consumption` | Monthly household consumption |
| `Savings` | Income minus consumption |
| `Household_Size` | Number of household members |
| `Urban` | 1 if urban, 0 if rural |
| `Region` | Rural or Urban label |

For our first regression, `Consumption` is the dependent variable and `Income` is the independent variable.

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
print("Number of households:", len(data))
print("Missing values by column:")
print(data.isna().sum())

data["Region"].value_counts()

## 4. Visual Intuition: Income and Consumption

A scatter plot should come before regression. It helps us see the basic pattern instead of blindly trusting model output.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(data["Income"], data["Consumption"])
plt.xlabel("Household Income")
plt.ylabel("Household Consumption")
plt.title("Income and Consumption")
plt.show()

### Discussion

1. Is the relationship positive or negative?
2. Is the relationship perfect?
3. Why might households with similar income have different consumption?

## 5. Correlation Example

Correlation measures association. It is useful, but it does not define a dependent variable and it does not prove causality.

In [ ]:
correlation = data["Income"].corr(data["Consumption"])
print("Correlation between income and consumption:", round(correlation, 3))

## 6. Simple Linear Regression with statsmodels

We estimate:

`Consumption = beta_0 + beta_1 Income + error`

`statsmodels` is useful because it provides econometric output such as coefficients, standard errors, p-values, and R-squared.

In [ ]:
X = data[["Income"]]
X = sm.add_constant(X)
Y = data["Consumption"]

model = sm.OLS(Y, X).fit()
model.summary()

## 7. Interpret the Regression Coefficients

The income coefficient is the main focus. It tells us the predicted change in consumption when income increases by one unit.

In [ ]:
intercept = model.params["const"]
income_coefficient = model.params["Income"]

print("Intercept:", round(intercept, 2))
print("Income coefficient:", round(income_coefficient, 4))
print()
print(f"When income increases by 1 rupee, predicted consumption increases by about {income_coefficient:.3f} rupees, on average.")
print(f"When income increases by 10,000 rupees, predicted consumption increases by about {income_coefficient * 10000:,.0f} rupees, on average.")

## 8. Visualize the Fitted Regression Line

The fitted line shows the model's predicted consumption values for the observed income levels.

In [ ]:
data["Predicted_Consumption"] = model.predict(X)

plt.figure(figsize=(8, 5))
plt.scatter(data["Income"], data["Consumption"], label="Actual households")
plt.plot(data["Income"], data["Predicted_Consumption"], color="red", label="Fitted regression line")
plt.xlabel("Household Income")
plt.ylabel("Household Consumption")
plt.title("Actual Consumption and Fitted Regression Line")
plt.legend()
plt.show()

## 9. Actual vs Predicted Values

A regression model produces predicted values. These predictions will usually not match the actual values perfectly.

In [ ]:
data[["Household", "Income", "Consumption", "Predicted_Consumption"]].head(10).round(2)

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(data["Household"], data["Consumption"], marker="o", label="Actual consumption")
plt.plot(data["Household"], data["Predicted_Consumption"], marker="o", label="Predicted consumption")
plt.xlabel("Household")
plt.ylabel("Consumption")
plt.title("Actual vs Predicted Consumption")
plt.legend()
plt.show()

## 10. Residuals and Prediction Errors

`Residual = Actual Consumption - Predicted Consumption`

Residuals help us identify where the model predicts poorly.

In [ ]:
data["Residual"] = data["Consumption"] - data["Predicted_Consumption"]

data[["Household", "Income", "Consumption", "Predicted_Consumption", "Residual"]].round(2).head(12)

In [ ]:
error_columns = ["Household", "Income", "Consumption", "Predicted_Consumption", "Residual"]

largest_positive = data.loc[[data["Residual"].idxmax()], error_columns]
largest_negative = data.loc[[data["Residual"].idxmin()], error_columns]

print("Largest positive residual:")
print(largest_positive.round(2).to_string(index=False))
print()
print("Largest negative residual:")
print(largest_negative.round(2).to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 5))
plt.axhline(0, color="black", linewidth=1)
plt.scatter(data["Income"], data["Residual"])
plt.xlabel("Income")
plt.ylabel("Residual")
plt.title("Residuals by Income")
plt.show()

## 11. Prediction Exercise

Use the regression equation to predict consumption for a household with income equal to 120,000.

In [ ]:
new_household = pd.DataFrame({"const": [1], "Income": [120000]})
predicted_consumption = model.predict(new_household)

print("Predicted consumption:", round(predicted_consumption.iloc[0], 2))

## 12. Prediction Error Metric

Mean absolute error gives the average size of prediction mistakes.

In [ ]:
mae = mean_absolute_error(data["Consumption"], data["Predicted_Consumption"])
print("Mean absolute error:", round(mae, 2))

## 13. Extending the Model: Household Size

The lecture emphasizes that real economic behavior is complex. Consumption may depend on income, but also on household size. Larger households may consume more even with similar income.

Now we estimate:

`Consumption = beta_0 + beta_1 Income + beta_2 Household_Size + error`

In [ ]:
X_multi = data[["Income", "Household_Size"]]
X_multi = sm.add_constant(X_multi)

model_multi = sm.OLS(Y, X_multi).fit()
model_multi.summary()

In [ ]:
print("Simple regression income coefficient:", round(model.params["Income"], 4))
print("Multiple regression income coefficient:", round(model_multi.params["Income"], 4))
print("Household size coefficient:", round(model_multi.params["Household_Size"], 2))

### Multiple Regression Interpretation

In multiple regression, the income coefficient means the predicted change in consumption when income changes, **holding household size constant**.

This phrase is central to regression interpretation.

## 14. Group Comparison: Urban and Rural Households

Regression should be connected to data understanding. Here we compare average income and consumption by region.

In [ ]:
region_summary = data.groupby("Region")[["Income", "Consumption", "Savings"]].mean()
region_summary.round(2)

In [ ]:
plt.figure(figsize=(8, 5))

for region, group in data.groupby("Region"):
    plt.scatter(group["Income"], group["Consumption"], label=region)

plt.xlabel("Income")
plt.ylabel("Consumption")
plt.title("Income and Consumption by Region")
plt.legend()
plt.show()

## 15. Prediction vs Causality

The model may predict consumption well, but that does not automatically prove that income alone causes consumption to change. There may be other factors such as household size, prices, credit access, expectations, or location.

The lecture example: ice cream sales and drowning incidents may move together because both rise during hot weather. The hidden factor is temperature.

In [ ]:
summer_example = pd.DataFrame({
    "Temperature": [24, 26, 28, 30, 32, 34, 36],
    "Ice_Cream_Sales": [120, 150, 190, 240, 300, 370, 450],
    "Drowning_Incidents": [1, 1, 2, 3, 4, 5, 6]
})

summer_example.corr(numeric_only=True).round(3)

## 16. statsmodels vs scikit-learn

`statsmodels` is often preferred for econometric interpretation. `scikit-learn` is often preferred for prediction workflows. Here we fit the same simple relationship using `scikit-learn`.

In [ ]:
sk_model = LinearRegression()
sk_model.fit(data[["Income"]], data["Consumption"])

sk_predictions = sk_model.predict(data[["Income"]])

print("scikit-learn intercept:", round(sk_model.intercept_, 2))
print("scikit-learn income coefficient:", round(sk_model.coef_[0], 4))
print("scikit-learn MAE:", round(mean_absolute_error(data["Consumption"], sk_predictions), 2))

## 17. Interactive Mini Quiz

Run these cells during the live session.

In [ ]:
answer = input("What is the main purpose of regression? A) Delete missing values B) Estimate relationships C) Make charts D) Rename columns: ")

if answer.upper() == "B":
    print("Correct. Regression estimates relationships between variables.")
else:
    print("Not quite. Regression is mainly used to estimate relationships between variables.")

In [ ]:
answer = input("In our first model, what is the dependent variable? A) Income B) Consumption C) Household_Size D) Region: ")

if answer.upper() == "B":
    print("Correct. Consumption is the outcome variable.")
else:
    print("Not quite. Consumption is the dependent variable in the first model.")

In [ ]:
answer = input("Does regression automatically prove causality? Yes/No: ")

if answer.lower() == "no":
    print("Correct. Regression alone does not automatically prove causality.")
else:
    print("Careful. Regression can estimate relationships and support prediction, but causality requires stronger reasoning and research design.")

## 18. Self-Assessment Tasks

Complete these tasks before submitting your Week 7 activity:

1. Estimate the simple regression between income and consumption.
2. Interpret the income coefficient in one clear sentence.
3. Create a scatter plot and fitted regression line.
4. Generate predicted consumption values.
5. Calculate residuals.
6. Identify the largest prediction errors.
7. Estimate the extended model using household size.
8. Explain why prediction is not the same as causality.

## Final Reflection

Write short answers:

1. What does the regression suggest about the relationship between income and consumption?
2. Why is the fitted line useful?
3. Why are residuals important?
4. What does it mean to hold household size constant?
5. Why must economists be careful when using regression output generated by AI tools?